## 1. Imports and paths

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

from pyBKT.models import Model

BASE_DIR = Path("..").resolve()          # app/
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"
MODELS_DIR = BASE_DIR / "models"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("MODELS_DIR:", MODELS_DIR)


BASE_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app
PROCESSED_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app\data\processed
OUTPUTS_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app\data\outputs
MODELS_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app\models


## 2. Load cleaned dataset

In [2]:
train_path = PROCESSED_DIR / "df_train_kc_cleaned.csv"
test_path = PROCESSED_DIR / "df_test_kc_cleaned.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print(df_train.columns.tolist())

Train shape: (409501, 12)
Test shape: (2808, 12)
['Anon Student Id', 'Problem Name', 'Step Name', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'new_KC']


## 3. Helpers and mapping

In [3]:
KC_NAME_MAP = {
    "Expand / Eliminate Parentheses": "expand_eliminate_parentheses",
    "Combine Like Terms": "combine_like_terms",
    "Move Constants Across the Equation": "move_constants",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

In [4]:
def prepare_columns(df):
    df = df.copy()

    time_cols = [
        "Step Start Time",
        "First Transaction Time",
        "Correct Transaction Time",
        "Step End Time"
    ]

    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df
def build_pybkt_df(df):
    df = df.copy()

    # Keep only rows with the core fields
    df = df.dropna(subset=[
        "Anon Student Id",
        "new_KC",
        "Correct First Attempt",
        "Step Start Time"
    ]).copy()

    # Keep only binary observations
    df["Correct First Attempt"] = pd.to_numeric(df["Correct First Attempt"], errors="coerce")
    df = df[df["Correct First Attempt"].isin([0, 1])].copy()

    # Keep only final KCs
    df = df[df["new_KC"].isin(KC_NAME_MAP.keys())].copy()

    # Build standard BKT columns
    df["user_id"] = df["Anon Student Id"].astype(str)
    df["skill_name"] = df["new_KC"].map(KC_NAME_MAP)
    df["correct"] = df["Correct First Attempt"].astype(int)

    # Sort chronologically
    df["Step Start Time"] = pd.to_datetime(df["Step Start Time"], errors="coerce")
    df = df.sort_values(
        by=["user_id", "Step Start Time", "Problem Name", "Step Name"]
    ).reset_index(drop=True)

    # Unique order_id
    df["order_id"] = np.arange(1, len(df) + 1)

    # Minimal pyBKT dataframe
    out = df[["order_id", "user_id", "skill_name", "correct"]].copy()

    return out

In [22]:
df_train = prepare_columns(df_train)
train_bkt = build_pybkt_df(df_train)

print("train_bkt shape:", train_bkt.shape)
print(train_bkt.dtypes)
print(train_bkt["skill_name"].value_counts())

train_bkt shape: (409067, 4)
order_id      int32
user_id         str
skill_name      str
correct       int32
dtype: object
skill_name
combine_like_terms              118858
remove_coefficient              118578
move_constants                  104716
expand_eliminate_parentheses     59476
move_variables_one_side           6505
normalize_negative_sign            934
Name: count, dtype: int64


{'order_id': 'order_id', 'user_id': 'user_id', 'skill_name': 'skill_name', 'correct': 'correct'}
     user_id  order_id               skill_name  correct
0    0I891Gg       225  normalize_negative_sign        1
1    0I891Gg       226  normalize_negative_sign        1
2   171017OL       956  normalize_negative_sign        1
3   171017OL       957  normalize_negative_sign        1
4   171051xl      1242  normalize_negative_sign        0
5   171051xl      1243  normalize_negative_sign        1
6   1712aRVk      2907  normalize_negative_sign        0
7   1712bs2B      3331  normalize_negative_sign        0
8   1713jr4f      5154  normalize_negative_sign        1
9   1715Zzr7      6395  normalize_negative_sign        1
10  1715Zzr7      6396  normalize_negative_sign        1
11  1715Zzr7      6401  normalize_negative_sign        1
12  1715Zzr7      6402  normalize_negative_sign        1
13  1716AHha      7355  normalize_negative_sign        1
14  1716AHha      7356  normalize_negative_sign 

c:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\.venv-pybkt\Lib\site-packages\pyBKT\fit\M_step.py:61: RuntimeWarning: invalid value encountered in divide
  model['pi_0'] = init_softcounts[:] / np.sum(init_softcounts[:])


                                          value
skill                   param   class          
normalize_negative_sign prior   default 0.20000
                        learns  default 0.15000
                        guesses default 0.20000
                        slips   default 0.10000
                        forgets default 0.00000
